# Réseau de neurones simple avec PyTorch Lightning

Dans ces travaux pratiques, nous construisons avec PyTorch Lightning des réseaux de neurones qui reconnaissent les chiffres manuscrits de MNIST. Lightning écrit la boucle d'entraînement à notre place : le modèle reste un `nn.Module` PyTorch, et un `Trainer` l'entraîne (slides « PyTorch Lightning », « Définir un `LightningModule` » et « Entraîner avec le `Trainer` »).

1. une régression logistique multinomiale, le réseau sans couche cachée, avec deux façons de donner les classes à la perte ;
2. un perceptron multicouche régularisé ;
3. une recherche d'hyper-paramètres avec Optuna ;
4. le suivi de l'entraînement avec TensorBoard, pour comparer deux optimiseurs.

Il n'est pas nécessaire d'utiliser une carte graphique pour ce TP : tout est rapide sur CPU. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import lightning
import matplotlib.pyplot as plt
import optuna
import torch
import torchmetrics
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Récupération des données

Comme dans le TP précédent, torchvision fournit le jeu MNIST.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Regardons les données

Étudiez comment sont stockées les données :

- la forme et le type des quatre tenseurs ;
- la valeur des premiers labels et l'intervalle des pixels ;
- quelques images, avec leur label ;
- le nombre d'exemples de chaque classe : les classes sont-elles équilibrées ?

**Documentation utile**

- [`torch.bincount`](https://docs.pytorch.org/docs/stable/generated/torch.bincount.html)
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()` et `X_train.max()`
- `torch.bincount(y_train)` : le nombre d'occurrences de chaque entier, de 0 à 9
- `plt.subplots(1, 10)` renvoie une figure et 10 axes ; `ax.imshow(image, cmap="gray_r")` affiche une image 28 × 28 en niveaux de gris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(X_train.dtype, X_train.min(), X_train.max(), y_train.dtype, y_train[:10])
print(torch.bincount(...))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(...)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
print(f"Format de X_train : {tuple(X_train.shape)}, de X_test : {tuple(X_test.shape)}")
print(f"Format de y_train : {tuple(y_train.shape)}, de y_test : {tuple(y_test.shape)}")
print(f"Type des images : {X_train.dtype}, de {X_train.min()} à {X_train.max()}")
print(f"Type des labels : {y_train.dtype}, premiers : {y_train[:10].tolist()}")
print(f"Exemples par classe : {torch.bincount(y_train).tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(int(y_train[i]))
  ax.axis("off")
plt.show()

- 60 000 images d'entraînement et 10 000 de test, de 28 × 28 pixels.
- Les images sont des entiers non signés sur 8 bits (`torch.uint8`), de 0 (fond) à 255 (encre) ; les labels, des entiers `torch.int64`, de 0 à 9.
- De 5 421 à 6 742 exemples par classe : des classes à peu près équilibrées.

## Transformation des données

Nous devons transformer ces données :

1. Aplatir les images : un tenseur de forme `(_, 784)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples. Chaque image devient un vecteur de 784 pixels, l'entrée attendue par une couche `nn.Linear`.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt que $[0, 255]$.
3. Encoder les classes en [*one-hot*](https://fr.wikipedia.org/wiki/Encodage_one-hot) : pour chaque exemple, 10 valeurs, toutes nulles sauf un 1 à l'indice de sa classe. C'est la première des deux façons de donner les classes à l'entropie croisée que nous verrons.

Effectuez les deux premières transformations sur `X_train` et `X_test`. Pour la troisième, gardez `y_train` et `y_test`, et rangez les classes encodées dans `Y_train` et `Y_test`.

**Documentation utile**

- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.reshape(-1, input_dim)` : `-1` laisse PyTorch calculer le nombre d'exemples
- `/ 255` divise chaque pixel et donne des `torch.float32`
- `nn.functional.one_hot(y_train, nb_classes)` renvoie un tenseur d'entiers de forme `(60000, 10)` ; `.float()` le convertit en flottants, le type qu'attend l'entropie croisée pour des cibles one-hot. Le label 5 y devient `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = X_train.reshape(...) / ...
X_test = ...
Y_train = nn.functional.one_hot(..., nb_classes).float()
Y_test = ...
```

### Solution

In [ ]:
# Des images « à plat » : l'entrée du réseau est un vecteur de input_dim pixels,
# dans [0, 1]. -1 laisse PyTorch calculer le nombre d'exemples.
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255

# Les classes en vecteurs one-hot, en flottants pour l'entropie croisée
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)
print(f"Label {int(y_train[0])} en one-hot : {Y_train[0].tolist()}")

## Le `LightningModule` du TP

La cellule suivante définit les outils des parties suivantes :

- `MNISTClassifier`, le `LightningModule` du TP (slide « Définir un `LightningModule` ») : il enveloppe un modèle PyTorch qui renvoie des *logits*, les scores bruts des 10 classes, et se charge de la perte, de la justesse (*accuracy*, avec torchmetrics) et de l'optimiseur. La perte est l'entropie croisée, qui applique elle-même la softmax (slide « Piège classique : softmax et entropie croisée ») et accepte comme cibles des vecteurs one-hot aussi bien que des indices de classe. L'optimiseur, son learning rate et sa *weight decay* sont des paramètres ;
- `get_dataloaders`, qui sépare des données en entraînement et validation (20 % par défaut) et renvoie leurs `DataLoader` ;
- `evaluate`, qui affiche la perte et la justesse d'un modèle entraîné sur l'ensemble de test.

In [ ]:
class MNISTClassifier(lightning.LightningModule):
  """Lightning wrapper: cross-entropy on the logits, accuracy as a metric."""

  def __init__(
    self,
    model: nn.Module,
    optimizer: type[torch.optim.Optimizer] = torch.optim.SGD,
    learning_rate: float = 0.01,
    weight_decay: float = 0.0,
    num_classes: int = nb_classes,
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, input_dim)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=num_classes
        )
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, targets = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, targets)
    # La métrique, elle, a besoin d'indices de classe
    labels = targets.argmax(dim=-1) if targets.dim() > 1 else targets
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimizer(
      self.parameters(),
      lr=self.hparams.learning_rate,
      weight_decay=self.hparams.weight_decay,
    )


def get_dataloaders(
  X: torch.Tensor, Y: torch.Tensor, batch_size: int = 128, validation_split: float = 0.2
) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation."""
  train_dataset, val_dataset = random_split(
    TensorDataset(X, Y), [1 - validation_split, validation_split]
  )
  return (
    DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    DataLoader(val_dataset, batch_size=batch_size),
  )


def evaluate(
  trainer: lightning.Trainer, classifier: MNISTClassifier, one_hot: bool = False
) -> None:
  """Affiche la perte et la justesse sur l'ensemble de test.

  one_hot : True pour un modèle entraîné sur les cibles one-hot (Y_train).
  """
  test_loader = DataLoader(
    TensorDataset(X_test, Y_test if one_hot else y_test), batch_size=128
  )
  score = trainer.test(classifier, test_loader, verbose=False)[0]
  print(f"Test : perte {score['test_loss']:.3f}, justesse {score['test_accuracy']:.3f}")

## Régression logistique multinomiale

Créez un modèle sans couche cachée, `model` : un `nn.Sequential` qui ne contient qu'une couche `nn.Linear` de `input_dim` entrées et `nb_classes` sorties. Chaque sortie est une combinaison linéaire des pixels, un logit, et la softmax de l'entropie croisée en fait 10 probabilités de somme 1 : c'est une régression logistique multinomiale.

Enveloppez-le dans `classifier`, un `MNISTClassifier` avec ses réglages par défaut. Affichez son résumé avec `ModelSummary`, et expliquez son nombre de paramètres.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(nn.Linear(input_dim, nb_classes))`
- `MNISTClassifier(model)` : SGD, avec un learning rate de 0,01
- `print(ModelSummary(classifier, max_depth=-1))` : `max_depth=-1` détaille toutes les couches, y compris celles du modèle enveloppé
- Résultat attendu : 7 850 paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model = nn.Sequential(nn.Linear(..., ...))
classifier = MNISTClassifier(...)
print(ModelSummary(..., max_depth=-1))
```

### Solution

In [ ]:
model = nn.Sequential(nn.Linear(input_dim, nb_classes))
classifier = MNISTClassifier(model)
print(ModelSummary(classifier, max_depth=-1))

# Une couche linéaire a un poids par entrée et par sortie, plus un biais par
# sortie : 784 × 10 + 10 = 7 850 paramètres

## Apprentissage

Entraînez `classifier` avec un `lightning.Trainer` (slide « Entraîner avec le `Trainer` ») :

- des batchs de 128 exemples, avec 20 % des données d'entraînement en validation : `get_dataloaders` sur `X_train` et les cibles one-hot `Y_train` ;
- 10 epochs ;
- un `CSVLogger`, qui enregistre les métriques dans `logs/simple`, et sans sauvegarde de checkpoints.

Mesurez enfin sa perte et sa justesse sur l'ensemble de test avec `evaluate`. Gardez ces chiffres : ils servent de référence aux modèles suivants.

**Documentation utile**

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.CSVLogger.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `get_dataloaders(X_train, Y_train, batch_size=128)` renvoie les `DataLoader` d'entraînement et de validation
- `lightning.Trainer(max_epochs=10, accelerator="auto", devices=1, logger=CSVLogger("logs", name="simple"), enable_checkpointing=False)` : `accelerator="auto"` prend le GPU s'il y en a un
- `callbacks=[TQDMProgressBar()]` : la barre de progression de tqdm, plutôt que celle de rich, que Lightning choisit quand rich est installé (Colab) et qui a figé des entraînements
- `trainer.fit(classifier, train_loader, val_loader)`
- `evaluate(trainer, classifier, one_hot=True)` : le modèle apprend sur des cibles one-hot
- Résultat attendu : environ 89 % de justesse en validation comme en test ; SGD, avec un learning rate de 0,01, progresse encore à la dixième epoch

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
train_loader, val_loader = get_dataloaders(..., ..., batch_size=...)
trainer = lightning.Trainer(
  max_epochs=...,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(..., ..., ...)
evaluate(trainer, classifier, one_hot=...)
```

### Solution

In [ ]:
# Le Trainer exécute la boucle d'entraînement complète du LightningModule
train_loader, val_loader = get_dataloaders(X_train, Y_train, batch_size=128)

trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="simple"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(classifier, train_loader, val_loader)
evaluate(trainer, classifier, one_hot=True)

## Une perte pour les labels entiers

L'entropie croisée de PyTorch accepte aussi, comme cibles, les labels entiers `y_train` (des indices de classe, de type `torch.int64`) : l'encodage one-hot devient inutile.

Créez `classifier2`, un `MNISTClassifier` qui enveloppe un nouveau modèle identique au précédent, et entraînez-le sur `y_train` avec les mêmes réglages (`CSVLogger` dans `logs/simple_sparse`). Comparez sa perte et sa justesse de test à celles du premier modèle : que change le format des cibles ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un nouveau `nn.Sequential(nn.Linear(input_dim, nb_classes))` : un modèle aux poids neufs
- `get_dataloaders(X_train, y_train, batch_size=128)`, puis un nouveau `Trainer`, avec `name="simple_sparse"`
- `evaluate(trainer2, classifier2)`, sans `one_hot`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
classifier2 = MNISTClassifier(nn.Sequential(nn.Linear(input_dim, nb_classes)))
train_loader2, val_loader2 = get_dataloaders(X_train, ..., batch_size=128)
trainer2 = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer2.fit(classifier2, ..., ...)
evaluate(trainer2, classifier2)
```

### Solution

In [ ]:
classifier2 = MNISTClassifier(nn.Sequential(nn.Linear(input_dim, nb_classes)))
train_loader2, val_loader2 = get_dataloaders(X_train, y_train, batch_size=128)

trainer2 = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="simple_sparse"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer2.fit(classifier2, train_loader2, val_loader2)
evaluate(trainer2, classifier2)

# Les mêmes résultats, aux tirages aléatoires près (initialisation, ordre des
# batchs) : seul le format des cibles change. Les labels entiers évitent de
# stocker 10 valeurs par exemple.

## Perceptron multicouche régularisé

Écrivez une fonction `build_deep_model` qui renvoie un perceptron multicouche neuf :

- 4 couches cachées de 20 neurones, avec l'activation ReLU ;
- pour toutes les couches, une initialisation orthogonale des matrices de poids (`nn.init.orthogonal_`) ;
- une couche de sortie de `nb_classes` neurones, sans activation : elle renvoie des logits, comme la régression logistique.

Enveloppez-en un dans `deep_classifier`, un `MNISTClassifier` qui l'entraîne avec l'optimiseur `torch.optim.AdamW`, un learning rate de 10⁻³ et une *weight decay* de 0,01 : la régularisation est portée par l'optimiseur (slide « AdamW et régularisation des poids »). Affichez son résumé, puis entraînez-le sur les labels entiers avec les mêmes réglages que la régression logistique. Combien de points de justesse en test gagne-t-il sur elle ? Pour combien de paramètres de plus ?

Les parties suivantes réutilisent `build_deep_model`.

**Documentation utile**

- [`torch.nn.init.orthogonal_`](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.orthogonal_)

In [ ]:
def build_deep_model() -> nn.Module: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.init.orthogonal_(layer.weight)` remplace sur place les poids d'une couche `nn.Linear` par une matrice orthogonale, qui conserve la norme du signal d'une couche à l'autre
- Une fonction `dense(in_features, out_features)` qui crée la couche et initialise ses poids évite de répéter ces deux lignes
- `nn.Sequential(dense(input_dim, 20), nn.ReLU(), ...)` : chaque couche cachée est suivie de son activation
- `MNISTClassifier(build_deep_model(), optimizer=torch.optim.AdamW, learning_rate=1e-3, weight_decay=0.01)`
- Résultat attendu : 17 170 paramètres, environ 95 % de justesse en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def dense(in_features: int, out_features: int) -> nn.Linear:
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(...)
  return layer


def build_deep_model() -> nn.Module:
  return nn.Sequential(
    dense(input_dim, 20),
    nn.ReLU(),
    ...,
    dense(20, nb_classes),
  )


deep_classifier = MNISTClassifier(
  build_deep_model(), optimizer=..., learning_rate=..., weight_decay=...
)
print(ModelSummary(deep_classifier, max_depth=-1))
train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)
deep_trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="deep"),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
deep_trainer.fit(..., ..., ...)
evaluate(deep_trainer, deep_classifier)
```

### Solution

In [ ]:
def dense(in_features: int, out_features: int) -> nn.Linear:
  """Une couche linéaire aux poids initialisés par une matrice orthogonale."""
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(layer.weight)
  return layer


def build_deep_model() -> nn.Module:
  """Un perceptron multicouche neuf : 4 couches cachées de 20 neurones."""
  return nn.Sequential(
    dense(input_dim, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, nb_classes),
  )


# La régularisation est portée par l'optimiseur : AdamW applique la weight decay
# directement aux poids, alors qu'une pénalité L2 ajoutée à la perte serait
# déformée par les pas adaptatifs d'Adam
deep_classifier = MNISTClassifier(
  build_deep_model(), optimizer=torch.optim.AdamW, learning_rate=1e-3, weight_decay=0.01
)
print(ModelSummary(deep_classifier, max_depth=-1))

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)
deep_trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="deep"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
deep_trainer.fit(deep_classifier, train_loader, val_loader)
evaluate(deep_trainer, deep_classifier)

# Environ 6 points de plus que la régression logistique (95 % contre 89 %), pour
# 17 170 paramètres au lieu de 7 850. Une partie du gain vient d'AdamW : la partie
# TensorBoard compare les deux optimiseurs sur ce modèle.

## Recherche d'hyper-paramètres

Combien de neurones dans la couche cachée, quel learning rate ? [Optuna](https://optuna.org/) essaie des combinaisons d'hyper-paramètres et garde la meilleure, notée sur l'ensemble de validation. Son échantillonneur par défaut, TPE, choisit chaque essai d'après les résultats des précédents (slide « Pour aller plus loin : recherche guidée et élagage »).

Écrivez une fonction objectif, `objective(trial)`, qui :

- tire les hyper-paramètres de l'essai : `units`, le nombre de neurones cachés, de 32 à 512 par pas de 32 ; `learning_rate`, le learning rate d'Adam, de 10⁻⁴ à 10⁻², en échelle logarithmique ;
- entraîne 3 fois un perceptron à une couche cachée ReLU, de sortie `nb_classes` logits, enveloppé dans un `MNISTClassifier` avec l'optimiseur `torch.optim.Adam`, sur les labels entiers : 10 epochs, des batchs de 1 500 exemples, 20 % de validation ;
- renvoie la moyenne des 3 justesses de validation finales, moins bruitée qu'une seule.

Créez ensuite une étude qui maximise cet objectif, lancez-la sur 5 essais, puis affichez le tableau des essais, les meilleurs hyper-paramètres et la meilleure justesse.

**Documentation utile**

- [`optuna.trial.Trial`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html)
- [`optuna.create_study`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html)
- [`optuna.study.Study.optimize`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html#optuna.study.Study.optimize)

In [ ]:
def objective(trial: optuna.Trial) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `trial.suggest_int("units", 32, 512, step=32)` et `trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)` renvoient la valeur de l'essai en cours
- `nn.Sequential(nn.Linear(input_dim, units), nn.ReLU(), nn.Linear(units, nb_classes))`
- `lightning.Trainer(..., logger=False, enable_progress_bar=False, enable_model_summary=False)` : un entraînement silencieux, pour que les 15 entraînements ne remplissent pas la sortie
- `trainer.callback_metrics["val_accuracy"]` : la justesse de validation de la dernière epoch, un tenseur
- `optuna.create_study(direction="maximize")` : une étude qui cherche le maximum de l'objectif ; `study.optimize(objective, n_trials=5)` la lance
- `study.trials_dataframe()`, `study.best_params` et `study.best_value` : les essais, les meilleurs hyper-paramètres et la meilleure valeur de l'objectif

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", ...)
  learning_rate = trial.suggest_float("learning_rate", ...)
  val_accuracies = []
  for _ in range(3):
    model = nn.Sequential(
      nn.Linear(input_dim, ...), nn.ReLU(), nn.Linear(..., nb_classes)
    )
    classifier = MNISTClassifier(model, optimizer=..., learning_rate=...)
    train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=...)
    trainer = lightning.Trainer(
      max_epochs=10,
      accelerator="auto",
      devices=1,
      logger=False,
      enable_checkpointing=False,
      enable_progress_bar=False,
      enable_model_summary=False,
    )
    trainer.fit(...)
    val_accuracies.append(float(trainer.callback_metrics[...]))
  return ...


study = optuna.create_study(direction=...)
study.optimize(objective, n_trials=...)
print(study.trials_dataframe())
print(study.best_params, study.best_value)
```

### Solution

In [ ]:
EXECUTIONS_PER_TRIAL = 3


def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", 32, 512, step=32)
  learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

  # Chaque configuration est entraînée plusieurs fois, l'objectif étant la
  # moyenne des justesses de validation obtenues
  val_accuracies = []
  for _ in range(EXECUTIONS_PER_TRIAL):
    model = nn.Sequential(
      nn.Linear(input_dim, units), nn.ReLU(), nn.Linear(units, nb_classes)
    )
    classifier = MNISTClassifier(
      model, optimizer=torch.optim.Adam, learning_rate=learning_rate
    )
    train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=1500)
    trainer = lightning.Trainer(
      max_epochs=10,
      accelerator="auto",
      devices=1,
      logger=False,
      enable_checkpointing=False,
      enable_progress_bar=False,
      enable_model_summary=False,
    )
    trainer.fit(classifier, train_loader, val_loader)
    val_accuracies.append(float(trainer.callback_metrics["val_accuracy"]))
  return sum(val_accuracies) / len(val_accuracies)


study = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
study.optimize(objective, n_trials=5)

In [ ]:
print(
  study.trials_dataframe()[["number", "value", "params_units", "params_learning_rate"]]
)
print(f"Meilleurs hyper-paramètres : {study.best_params}")
print(f"Meilleure justesse de validation : {study.best_value:.4f}")

Le learning rate compte plus que le nombre de neurones. Dans une exécution, le meilleur essai (288 neurones, learning rate de 9,4·10⁻⁴) atteint 95,3 % de justesse de validation, quand les essais au learning rate proche de 10⁻⁴ restent entre 88,5 % et 90,5 %, quelle que soit leur taille : en 10 epochs, un pas si petit n'a pas le temps de converger.

Avec 5 essais, TPE ne guide pas encore la recherche : ses 10 premiers essais sont tirés au hasard (le paramètre `n_startup_trials` de `optuna.samplers.TPESampler`), le temps de réunir assez de résultats pour modéliser le score. Une recherche guidée ne vaut donc que pour des dizaines d'essais.

## AdamW contre SGD sous TensorBoard

TensorBoard affiche les courbes d'apprentissage qu'un [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html), passé au `Trainer` par son argument `logger`, écrit dans un dossier : un sous-dossier par entraînement. Lancez la cellule suivante avant l'exercice : le tableau de bord se remplit au fil des epochs (bouton de rechargement en haut à droite).

Entraînez ensuite deux perceptrons multicouches neufs (`build_deep_model()`) avec les réglages de la partie « Perceptron multicouche régularisé », l'un avec `torch.optim.AdamW` et un learning rate de 10⁻³, l'autre avec `torch.optim.SGD` et un learning rate de 0,01, tous deux avec une *weight decay* de 0,01. Passez à chaque `Trainer` un `TensorBoardLogger` qui écrit dans `logs/adamw` pour le premier, dans `logs/sgd` pour le second.

Comparez les courbes `val_loss` et `val_accuracy` des deux entraînements : lequel converge le plus vite ? Pourquoi ?

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `TensorBoardLogger("logs", name="adamw")` : les courbes vont dans `logs/adamw`
- Un modèle neuf par optimiseur, `build_deep_model()` : sinon le second entraînement repart des poids du premier
- `train_loader` et `val_loader`, les `DataLoader` de la partie « Perceptron multicouche régularisé », servent aux deux entraînements
- Dans TensorBoard, l'onglet *Scalars* superpose les courbes des deux dossiers ; `CSVLogger` écrit aussi dans `logs`, mais TensorBoard ignore ses fichiers
- Relancer un entraînement crée un nouveau sous-dossier `version_<n>` : supprimez `logs` pour repartir d'un tableau de bord propre

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
optimizers = {"adamw": (torch.optim.AdamW, 1e-3), "sgd": (torch.optim.SGD, 0.01)}
for name, (optimizer, learning_rate) in optimizers.items():
  classifier = MNISTClassifier(
    build_deep_model(),
    optimizer=...,
    learning_rate=...,
    weight_decay=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=...),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
```

### Solution

In [ ]:
optimizers = {"adamw": (torch.optim.AdamW, 1e-3), "sgd": (torch.optim.SGD, 0.01)}
for name, (optimizer, learning_rate) in optimizers.items():
  classifier = MNISTClassifier(
    build_deep_model(),
    optimizer=optimizer,
    learning_rate=learning_rate,
    weight_decay=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=name),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
  print(f"{name} : val_accuracy {float(trainer.callback_metrics['val_accuracy']):.3f}")

# AdamW converge bien plus vite : 91 % de justesse de validation dès la première
# epoch et 95 % à la dixième, quand SGD part de 42 % et finit vers 90 %. SGD fait un
# pas de même taille (0,01) pour tous les poids ; Adam adapte le pas de chaque poids
# à la taille de ses gradients récents, et garde une inertie d'un pas à l'autre (la
# moyenne des gradients).